In [2]:
import numpy as np
from osgeo import gdal
import os
import dask.array as da
import rioxarray as rxr
import pandas as pd

In [3]:
def align_raster_to_reference_2(input_raster, reference_raster, output_raster):
    """
    Align input_raster to the exact grid of reference_raster.
    Ensures shape, pixel size, extent, projection all match.
    """

    ref = gdal.Open(reference_raster)
    print("GeoTransform:", ref.GetGeoTransform())
    print("Size:", ref.RasterXSize, ref.RasterYSize)
    
    if ref is None:
        raise FileNotFoundError(f"Cannot open reference raster: {reference_raster}")

    ref_gt = ref.GetGeoTransform()
    ref_proj = ref.GetProjection()
    xsize = ref.RasterXSize
    ysize = ref.RasterYSize

    # Compute bounds from geotransform
    minx = ref_gt[0]
    maxy = ref_gt[3]
    maxx = minx + xsize * ref_gt[1]
    miny = maxy + ysize * ref_gt[5]

    warp_opts = gdal.WarpOptions(
        format="GTiff",
        dstSRS=ref_proj,
        outputBounds=(minx, miny, maxx, maxy),
        width=xsize,
        height=ysize,
        resampleAlg="nearest",
        creationOptions=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BLOCKXSIZE=256",
            "BLOCKYSIZE=256"
        ]
    )

    gdal.Warp(
        output_raster,
        input_raster,
        options=warp_opts
    )

    ref = None

def mask_raster_by_values(raster1_path, raster2_path, output_path, mask_values=(1, 6), nodata_value=0):

    # --- Open reference raster2 ---
    r2 = gdal.Open(raster2_path)
    if r2 is None:
        raise FileNotFoundError(f"Cannot open raster2: {raster2_path}")

    r2_arr = r2.GetRasterBand(1).ReadAsArray()
    r2_shape = r2_arr.shape

    # --- Open raster1 ---
    r1 = gdal.Open(raster1_path)
    if r1 is None:
        raise FileNotFoundError(f"Cannot open raster1: {raster1_path}")

    r1_arr = r1.GetRasterBand(1).ReadAsArray()

    # =====================================================================
    # ALIGN IF SHAPES DO NOT MATCH
    # =====================================================================
    if r1_arr.shape != r2_shape:

        print("Shapes differ → aligning raster1 to raster2.")
        print(f"  raster1 shape: {r1_arr.shape}")
        print(f"  raster2 shape: {r2_shape}")

        tmp_aligned = raster1_path.replace(".tif", "") + "_aligned_tmp.tif"

        # run alignment (your function)
        align_raster_to_reference_2(raster1_path, raster2_path, tmp_aligned)

        print("The data is aligned")
        # reopen aligned raster
        r1_aligned = gdal.Open(tmp_aligned)
        if r1_aligned is None:
            raise RuntimeError("Alignment failed: Temporary aligned raster not created.")

        r1_arr = r1_aligned.GetRasterBand(1).ReadAsArray()

        # CLOSE the aligned dataset before deleting the file
        r1_aligned = None

        # strict validation
        if r1_arr.shape != r2_shape:
            os.remove(tmp_aligned)
            raise ValueError(
                f"Alignment FAILED: aligned raster1 has shape {r1_arr.shape}, "
                f"but raster2 has shape {r2_shape}"
            )

        # delete temp file safely
        os.remove(tmp_aligned)

    # =====================================================================
    # APPLY MASK
    # =====================================================================
    print("Applying mask from coral to wetlands...")
    mask = np.isin(r1_arr, mask_values)

    output_arr = r2_arr.copy()
    output_arr[mask] = nodata_value

    # =====================================================================
    # WRITE OUTPUT
    # =====================================================================
    print("Saving output...")
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        output_path,
        r2.RasterXSize,
        r2.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BLOCKXSIZE=256",
            "BLOCKYSIZE=256"
        ]
    )

    out_ds.SetGeoTransform(r2.GetGeoTransform())
    out_ds.SetProjection(r2.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(output_arr)
    out_band.SetNoDataValue(nodata_value)
    out_band.FlushCache()

    # close output
    out_ds = None

    print(f"Masked raster written to:\n  {output_path}")


def mask_raster_by_values_dask(raster1_path, raster2_path, output_path, mask_values=(1, 6), nodata_value=0, chunks=(2048, 2048)):
    """
    Memory-safe raster masking using Dask.
    Automatically handles huge rasters.
    """

    print("Opening rasters with Dask...")

    # Open as lazy Dask arrays
    r1 = rxr.open_rasterio(raster1_path, chunks=chunks)
    r2 = rxr.open_rasterio(raster2_path, chunks=chunks)

    # Remove band dimension if single-band
    r1 = r1.squeeze()
    r2 = r2.squeeze()

    # =====================================================================
    # ALIGNMENT CHECK
    # =====================================================================
    
    # Convert affine to GDAL-style 6-element tuple
    t1 = r1.rio.transform().to_gdal()
    t2 = r2.rio.transform().to_gdal()
    
    if r1.rio.shape != r2.rio.shape or not np.allclose(t1, t2):
        
        df_error = pd.DataFrame({
        "r1.rio.shape": [r1.rio.shape],
        "r2.rio.shape": [r2.rio.shape],
        "r1.rio.transform()": [r1.rio.transform()],
        "r2.rio.transform()": [r2.rio.transform()]
        })
        
        print("Rasters are not aligned. Returning mismatch info.")
        return df_error
        
        
    print("Building mask lazily...")

    # Convert mask_values to Dask-friendly operation
    mask = da.isin(r1.data, list(mask_values))

    print("Applying mask...")

    # Apply mask lazily
    result = da.where(mask, nodata_value, r2.data)

    # Wrap back into xarray
    result_xr = r2.copy(data=result)
    result_xr.rio.write_nodata(nodata_value, inplace=True)

    print("Writing output (this is where computation happens)...")

    # Write to disk (triggers computation)
    result_xr.rio.to_raster(
        output_path,
        tiled=True,
        compress="DEFLATE",
        blockxsize=256,
        blockysize=256,
        windowed=True
    )

    return print(f"Done → {output_path}")

In [ ]:
mask_raster_by_values(
    raster1_path=r"Z:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\coral_systems_4326.tif",
    raster2_path=r"Z:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\global_wetlands_100m_2020.tif",
    output_path=r"Z:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\global_wetlands_100m_2020_masked.tif",
    mask_values=(1, 6),
    nodata_value=0
)

In [8]:
raster1_path=r"X:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\coral_systems_4326.tif"
raster2_path=r"X:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\global_wetlands_100m_2021.tif"
output_path=r"X:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\global_wetlands_100m_2021_masked.tif"

# delete temp file safely
aligned_r1 = raster1_path.replace(".tif", "") + "_aligned_tmp.tif"

In [9]:
align_raster_to_reference_2(
    input_raster=raster1_path,
    reference_raster=raster2_path,
    output_raster=aligned_r1
)

GeoTransform: (-180.0, 0.0009, 0.0, 80.0, 0.0, -0.0009)
Size: 400000 155556


In [10]:
df_error = mask_raster_by_values_dask(
    raster1_path=aligned_r1,
    raster2_path=raster2_path,
    output_path=output_path,
    mask_values=(1, 6),
    nodata_value=0
)

Opening rasters with Dask...
Building mask lazily...
Applying mask...
Writing output (this is where computation happens)...


KeyboardInterrupt: 

In [30]:
df_error

,r1.rio.shape,r2.rio.shape,r1.rio.transform(),r2.rio.transform()
0,"(155556, 400000)","(155556, 400000)","(0.0009, 0.0, -180.0, 0.0, -0.0008999999999999...","(0.0009, 0.0, -180.0, 0.0, -0.0009, 80.0, 0.0,..."


In [ ]:
os.remove(aligned_r1)